# Phase 4 -- Notebook 2: B and A (S3, S4)

Independent of the other notebook: downloads the Phase 4 data itself and needs no other notebook's output.

| Run | Phase-1 model | Method | Submit? |
|---|---|---|---|
| **S3** | B (122M, 24 layers) | **AdaLoRA** (own code: rank budget 12 -> 8 per layer, moved to where importance is highest) | yes |
| **S4** | A (163M) | **Gradual unfreezing** (layer-wise fine-tuning: top 2 -> 8 of 16 blocks open over training, lower blocks get smaller learning rates) | yes |
| R3 | B | LoRMA -- reference: AdaLoRA vs LoRMA on the same model | no (dev only) |
| TF-IDF | none | Word 1-2-gram TF-IDF + logistic regression on CPU -- reference: how much does Phase 1 pretraining add? | no (dev only) |

**Pipeline (every run)** -- cases are long, so each is cut into 512-token chunks (64-token overlap; the first
chunk + up to 15 from the end).
1. **Adapt** the Phase 1 model with its method on one chunk per case per step (the last chunk half the time,
   otherwise a random one), labelled with the case outcome, on 80% of train.
2. **Encode** every chunk of the held-out 20% of train, dev and test with the adapted model.
3. **Read the whole case**: 7 small readers over the chunk vectors + chunk positions (last chunk, first chunk,
   mean, max, attention, GRU + attention, transformer) and a section study (first / middle / last third, last 4
   chunks), trained on the held-out 20% only and chosen on a validation slice of it -- dev stays clean.
4. **Calibrate**: decision threshold fitted on dev, used only if a 2-fold dev check agrees.

**How to run**
1. Inputs: `legallm-tokenizers` (for `t1_bpe.json`) and the datasets holding the **A** and **B** checkpoints (file names keep their `A_step... / B_step... / C_step... / D_step...` start;
   `.pt`, `.zip` or a folder Kaggle unzipped are all fine).
2. Secret on: `HF`. Accelerator **GPU T4 x2**, **internet on**.
3. **Save Version -> Save & Run All.** Time: ~4 h (S3 | S4, then R3; the TF-IDF reference runs on CPU alongside).
4. Submit `p4_nb2/S3/submission.zip`, `p4_nb2/S4/submission.zip`. Download `p4_nb2/ens_nb2.json` too
   (dev/test scores of every run, for the ensemble submission S6).


## 1. Inputs: tokenizer and checkpoints

In [ ]:
import csv
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

import numpy as np
import torch

N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p4_nb2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / "ckpts"
CKPT_DIR.mkdir(exist_ok=True)


def step_of(p):
    return int(Path(p).name.split("_step")[1].split("_")[0].split(".")[0])


def _rebuild_from_folder(folder, dest):
    """Kaggle unzips an uploaded checkpoint (a .pt file is a zip archive inside),
    leaving a folder with data.pkl, data/0, version, ... Re-pack it into a .pt."""
    pkl = next(Path(folder).rglob("data.pkl"))
    root = pkl.parent
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    return dest


def _collect(prefix):
    """{name: path} of usable <prefix>_step*.pt checkpoints under /kaggle/input:
    a .pt file; a .zip holding the .pt; the .pt renamed .zip; or a folder that
    Kaggle unzipped from an uploaded checkpoint (re-packed here)."""
    found = {}
    if not KAGGLE_INPUT.exists():
        return found
    for p in KAGGLE_INPUT.rglob(f"{prefix}_step*"):
        stem = p.name[:-3] if p.name.endswith(".pt") else (p.name[:-4] if p.name.endswith(".zip") else p.name)
        dest = CKPT_DIR / (stem + ".pt")
        if p.is_file() and p.suffix == ".pt":
            found.setdefault(p.name, p)
        elif p.is_dir() and any(p.rglob("data.pkl")) and dest.name not in found:
            found[dest.name] = _rebuild_from_folder(p, dest)
            print(f"  re-packed unzipped checkpoint folder {p} -> {dest.name}")
    for z in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                inner = [n for n in names if Path(n).name.startswith(f"{prefix}_step") and n.endswith(".pt")]
                for n in inner:
                    dest = CKPT_DIR / Path(n).name
                    if dest.name not in found:
                        with zf.open(n) as src, open(dest, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        found[dest.name] = dest
                if not inner and z.name.startswith(f"{prefix}_step") and any(n.endswith("data.pkl") for n in names):
                    dest = CKPT_DIR / (z.stem + ".pt")
                    if dest.name not in found:
                        shutil.copy(z, dest)
                        found[dest.name] = dest
        except zipfile.BadZipFile:
            continue
    return found




def checkpoints(prefix):
    found = _collect(prefix)
    assert found, f"No {prefix}_step* checkpoint (.pt, .zip or unzipped folder) under /kaggle/input -- attach it."
    return max(found.values(), key=step_of)


def find_one(name, label):
    hits = sorted(KAGGLE_INPUT.rglob(name)) if KAGGLE_INPUT.exists() else []
    assert hits, f"{name} not found under /kaggle/input -- attach {label}."
    return hits[0]


TOK_PATH = find_one("t1_bpe.json", "legallm-tokenizers")
CKPT = {"A": str(checkpoints("A")), "B": str(checkpoints("B"))}
for k, p in CKPT.items():
    print(f"{k}: {Path(p).name}")


## 2. Code files (written to /kaggle/working)

In [ ]:
%%writefile /kaggle/working/a_model.py
"""a_model.py -- Model A (Llama-style decoder) classes, copied from p1_03_model_A.ipynb."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction -- cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def _forward_inner(self, x, cos, sin, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, cache=None, use_checkpoint=False):
        # Checkpointing needs a cache-free forward (no in-place cache
        # mutation across a recomputed backward pass) and only makes
        # sense during training, never during KV-cache generation.
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, cache)


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values."""
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: raw hidden states, no LM head. attention_mask
        is accepted for interface compatibility with later phases (e.g. a
        padded batch) but unused here -- Phase 1 pretraining always packs
        full, unpadded windows, so every position is valid.
        """
        assert mode == "causal", "MiniGPT is decoder-only; only causal encoding is supported."
        B, T = input_ids.shape
        x = self.embed(input_ids)
        cos = self.rope_cos[:T].to(x.device)
        sin = self.rope_sin[:T].to(x.device)
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, cache=None, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, caches=None):
        if caches is None:
            return self.head(self.encode(ids))
        # Generation path (KV cache present): identical to encode() but
        # threading a per-layer cache through and offsetting RoPE by how
        # much of the cache is already filled.
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache=caches[i], use_checkpoint=False)
        x = self.norm_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", top_p=0.9, temperature=1.0):
        """Contract method. `sampler`: "greedy" (argmax, deterministic --
        what the smoke-test submission uses) or "top_p" (nucleus sampling).
        Stops early (per sequence, via a done mask) at <eos> or max_seq_len.
        """
        assert kv_cache, "This notebook's generate() only implements the KV-cache path -- it's what training will actually ship."
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        # A prompt longer than the model's context has nowhere to put
        # max_new_tokens (or, if it's longer than max_seq_len outright,
        # RoPE's precomputed cos/sin tables don't even cover it -- a hard
        # crash, not a quality issue). Truncate to the most recent tokens
        # -- the ones closest to where generation continues -- rather
        # than letting this crash the whole generation loop over one long
        # dev-set example.
        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            print(f"  [generate() warning: prompt has {input_ids.shape[1]} tokens, "
                  f"truncating to the last {max_prompt_len} to leave room for "
                  f"{max_new_tokens} new tokens within max_seq_len={self.config['max_seq_len']}]")
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])

        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)  # prefill: process the whole prompt at once
        next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all():
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)  # one new token at a time
            next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        self.train()
        return generated

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        mask = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * mask
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function: the one place that turns a config dict into a
    model instance, so every later phase constructs Model A the same way.
    """
    return MiniGPT(config)


USE_GRAD_CHECKPOINTING = True
EOS_ID = None


In [ ]:
%%writefile /kaggle/working/b_model.py
"""b_model.py -- Model B (local/global decoder) classes, copied from p1_04_model_B.ipynb."""
import math
Z_LOSS_COEF = 1e-4
EOS_ID = None
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


_TRAIN_MASK_CACHE = {}


def sliding_window_mask(q_start, n_q, n_k, window, device, cacheable=False):
    """Boolean [n_q, n_k] mask, True = may attend. Query i sits at absolute
    position q_start + i; key j at position j. Allowed iff the key is not in
    the future AND is less than `window` positions back.
    Only training masks are cached (training always uses one fixed shape);
    eval/generation shapes vary per prompt, so caching them would leak memory.
    """
    cache_key = (n_q, n_k, window, str(device))
    if cacheable and cache_key in _TRAIN_MASK_CACHE:
        return _TRAIN_MASK_CACHE[cache_key]
    q_pos = torch.arange(q_start, q_start + n_q, device=device)[:, None]
    k_pos = torch.arange(n_k, device=device)[None, :]
    allowed = (k_pos <= q_pos) & (q_pos - k_pos < window)
    if cacheable:
        _TRAIN_MASK_CACHE[cache_key] = allowed
    return allowed


class LocalGlobalAttention(nn.Module):
    """MQA attention. Local layers: RoPE + sliding-window mask. Global
    layers: no RoPE (NoPE), plain causal attention over everything.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, use_rope, qk_norm):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.use_rope = use_rope
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)
        self.q_norm = RMSNorm(d_head) if qk_norm else None
        self.k_norm = RMSNorm(d_head) if qk_norm else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        # The cache keeps the single KV head (that's MQA's memory saving);
        # expand it to all query heads only for the attention call itself.
        # An explicit copy keeps SDPA on its fast memory-efficient kernel.
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            # Global layer. Training / prefill: square causal. Decode step
            # (one query): may see every cached key -> no mask needed.
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg, is_global):
        super().__init__()
        self.is_global = is_global
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = LocalGlobalAttention(cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"],
                                         use_rope=not is_global, qk_norm=cfg["qk_norm"])
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg["d_model"], cfg["d_ff"])

    def _forward_inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, attn_mask, cache)


class KVCache:
    """Pre-allocated per-layer cache. Keeps the full history (with 1 KV
    head it's only ~0.5 MB per layer at 2048 tokens); the sliding window is
    applied by the attention mask, not by evicting entries.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


def is_global_layer(layer_idx, global_every):
    return (layer_idx + 1) % global_every == 0


class LocalGlobalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config, is_global=is_global_layer(i, config["global_every"]))
            for i in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # tied embeddings

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def layer_pattern(self):
        return "".join("G" if b.is_global else "L" for b in self.blocks)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def _run_blocks(self, x, pos_offset, caches, use_ckpt):
        T = x.shape[1]
        assert pos_offset + T <= self.config["max_seq_len"], "Sequence longer than max_seq_len."
        cos = self.rope_cos[pos_offset:pos_offset + T]
        sin = self.rope_sin[pos_offset:pos_offset + T]
        local_mask = sliding_window_mask(pos_offset, T, pos_offset + T, self.config["window"], x.device,
                                         cacheable=self.training and caches is None)
        for i, blk in enumerate(self.blocks):
            mask = None if blk.is_global else local_mask
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, mask, cache=cache, use_checkpoint=use_ckpt)
        return x

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: hidden states, no LM head. attention_mask is
        accepted for interface compatibility (Phase 1 packs full windows)."""
        assert mode == "causal", "Model B is decoder-only; only causal encoding is supported."
        x = self.embed(input_ids)
        x = self._run_blocks(x, 0, None, use_ckpt=self.training and USE_GRAD_CHECKPOINTING)
        return self.norm_f(x)

    def forward(self, ids, targets=None, caches=None):
        """No targets -> logits. With targets -> tensor [1, 2] = [cross-entropy,
        cross-entropy + z-loss]. The [1, 2] shape is so nn.DataParallel can
        stack one row per GPU; callers take .mean(0).
        In training mode, fp16 autocast is switched on here (inside the
        module) so it also applies inside DataParallel's worker threads.
        """
        use_amp = self.training and ids.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            if caches is None:
                h = self.encode(ids)
            else:
                pos_offset = caches[0].pos
                assert ids.shape[1] == 1 or pos_offset == 0, "Prefill must start from an empty cache."
                h = self.norm_f(self._run_blocks(self.embed(ids), pos_offset, caches, use_ckpt=False))
            logits = self.head(h)
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        target_logit = logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        ce = (lse - target_logit).mean()           # identical to F.cross_entropy
        total = ce + Z_LOSS_COEF * lse.pow(2).mean()
        return torch.stack([ce, total]).unsqueeze(0)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy",
                 top_p=0.9, temperature=1.0, repetition_penalty=1.0):
        """Contract method. Stops per sequence at <eos>. Prompts too long to
        leave room for max_new_tokens are cut to their most recent tokens."""
        assert kv_cache, "Only the KV-cache path is implemented."
        was_training = self.training
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])
        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)[:, -1, :]
        next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)
        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all() or generated.shape[1] >= max_total:
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)[:, -1, :]
            next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)

        self.train(was_training)
        return generated

    @staticmethod
    def _penalize(logits, previous_ids, penalty):
        """CTRL-style repetition penalty on every token already present."""
        if penalty == 1.0:
            return logits
        scores = logits.gather(1, previous_ids)
        scores = torch.where(scores > 0, scores / penalty, scores * penalty)
        return logits.scatter(1, previous_ids, scores)

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        keep = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function."""
    return LocalGlobalGPT(config)


USE_GRAD_CHECKPOINTING = True




In [ ]:
%%writefile /kaggle/working/cj_lib.py
"""cj_lib.py -- Phase 4 (Court Judgment Prediction, CJPE) library shared by both notebooks.

Pipeline (every experiment):
  case text -> Phase-1 tokens -> 512-token chunks with 64-token overlap (the first chunk
  plus up to MAX_CHUNKS-1 chunks from the end of the case).
  Stage 1  ADAPT   Phase-1 backbone + adaptation method + attention pooling + 1 logit,
                   trained on ONE chunk per case per step (the last chunk half of the time,
                   otherwise a random chunk), labelled with the case outcome, on 80% of train.
  Stage 2  ENCODE  the adapted backbone turns every chunk of the held-out 20% of train,
                   dev and test into a vector (cached in memory).
  Stage 3  READ    small aggregators over the chunk vectors + chunk positions, trained on the
                   held-out 20% only (the encoder never trained on those cases, so no stacking
                   leakage); compared on a validation slice of that 20%. The long-document and
                   section study (head / middle / end / whole case; mean, max, attention, GRU,
                   transformer) costs seconds here.
  CALIBRATE        decision threshold fitted on dev, used only if a 2-fold dev check agrees.

Methods (own code): LoRMA (Bihany et al. 2025, additive-identity form), AdaLoRA (Zhang et al.
2023), BitFit (Ben Zaken et al. 2022), gradual unfreezing with layer-wise learning rates
(Howard & Ruder 2018), full fine-tuning.
Distillation: hierarchical student (C cut to its first layers, reads the last 4 chunks) learns
gold labels + the teacher ensemble's probabilities + the teachers' chunk attention
("where to look").
"""
import csv
import json
import math
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CHUNK = 512
STRIDE = 448                     # 64-token overlap
MAX_CHUNKS = 16                  # first chunk + 15 from the end
MAX_TOKENS = CHUNK + (CHUNK + (MAX_CHUNKS - 2) * STRIDE)     # tokens kept per case
KD_LAST = 4                      # chunks the student reads


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_json_any(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


def parse_cases(obj):
    """Accepts a list of case dicts, a {id: case} dict, or {"data": [...]}.
    -> list of {"id": str, "raw_id": as given, "text": str, "label": 0/1 or None}."""
    if isinstance(obj, dict):
        for k in ("data", "cases", "documents", "examples", "items"):
            if k in obj and isinstance(obj[k], list):
                return parse_cases(obj[k])
        items = []
        for k, v in obj.items():
            items.append(dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v})
        return parse_cases(items)
    out = []
    for i, v in enumerate(obj):
        raw = v.get("id", v.get("case_id", v.get("name", i)))
        text = v.get("text", v.get("case_text", v.get("document", v.get("judgment", ""))))
        if isinstance(text, list):
            text = "\n".join(str(t) for t in text)
        lab = v.get("label", v.get("labels", v.get("decision", v.get("outcome"))))
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        if lab is not None:
            lab = int(float(lab))
            assert lab in (0, 1), f"case {raw}: label {lab} is not 0/1"
        out.append({"id": str(raw), "raw_id": raw, "text": " ".join(str(text or "").split()), "label": lab})
    return out


def windows(n, size, stride):
    """Start/end pairs covering 0..n with overlapping windows (last one ends at n)."""
    if n <= size:
        return [(0, n)]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return [(s, s + size) for s in starts]


def case_chunks(toks_len, n_orig, cut):
    """Chunk spans over the kept tokens: (start, end, start_frac, end_frac) where the
    fractions are positions in the ORIGINAL case (0 = first token, 1 = last)."""
    def orig(i):
        return i if (not cut or i < CHUNK) else n_orig - (toks_len - i)
    if cut:
        spans = [(0, CHUNK)] + [(s + CHUNK, e + CHUNK) for s, e in windows(toks_len - CHUNK, CHUNK, STRIDE)]
    else:
        spans = windows(toks_len, CHUNK, STRIDE)
    return [(s, e, orig(s) / n_orig, (orig(e - 1) + 1) / n_orig) for s, e in spans]


def tokenize_cases(cases, encode_batch, fallback_id, batch=256, log=print):
    """encode_batch(list of str) -> list of token-id lists. Keeps the first CHUNK tokens and
    the last MAX_TOKENS - CHUNK tokens of long cases."""
    out, t0 = [], time.time()
    for i in range(0, len(cases), batch):
        part = cases[i:i + batch]
        toks = encode_batch([c["text"] for c in part])
        for c, t in zip(part, toks):
            t = list(t) or [fallback_id]
            n = len(t)
            cut = n > MAX_TOKENS
            kept = t[:CHUNK] + t[n - (MAX_TOKENS - CHUNK):] if cut else t
            arr = np.asarray(kept, dtype=np.int64)
            assert arr.max() < 65536, "token id does not fit uint16"
            out.append({"id": c["id"], "raw_id": c["raw_id"], "label": c["label"], "toks": arr.astype(np.uint16),
                        "n_tok": n, "cut": cut, "chunks": case_chunks(len(kept), n, cut)})
        if (i // batch) % 40 == 0:
            log(f"  tokenized {min(i + batch, len(cases))}/{len(cases)} ({time.time() - t0:.0f}s)")
    return out


def split_train(cases, frac_agg=0.2, frac_val=0.15, seed=7):
    """Stratified split of train into adapt (stage 1) / agg_train / agg_val (stage 3)."""
    rng = random.Random(seed)
    adapt, agg_tr, agg_val = [], [], []
    for lab in (0, 1):
        ids = [c["id"] for c in cases if c["label"] == lab]
        rng.shuffle(ids)
        n_agg = int(round(frac_agg * len(ids)))
        n_val = int(round(frac_val * n_agg))
        agg_val += ids[:n_val]
        agg_tr += ids[n_val:n_agg]
        adapt += ids[n_agg:]
    return {"adapt": sorted(adapt), "agg_train": sorted(agg_tr), "agg_val": sorted(agg_val)}


# ---------------------------------------------------------------------------
# Metrics and calibration
# ---------------------------------------------------------------------------
def binary_report(y, pred):
    y, p = np.asarray(y, int), np.asarray(pred, int)
    rep = {}
    for c in (0, 1):
        tp = int(((p == c) & (y == c)).sum())
        fp = int(((p == c) & (y != c)).sum())
        fn = int(((p != c) & (y == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        rep[c] = (prec, rec, 2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return {"macro_f1": round((rep[0][2] + rep[1][2]) / 2, 4),
            "macro_precision": round((rep[0][0] + rep[1][0]) / 2, 4),
            "macro_recall": round((rep[0][1] + rep[1][1]) / 2, 4),
            "f1_0": round(rep[0][2], 4), "f1_1": round(rep[1][2], 4),
            "precision_1": round(rep[1][0], 4), "recall_1": round(rep[1][1], 4),
            "accuracy": round(float((y == p).mean()) if len(y) else 0.0, 4)}


def macro_f1(y, pred):
    return binary_report(y, pred)["macro_f1"]


def roc_auc(y, s):
    y, s = np.asarray(y, int), np.asarray(s, float)
    n1, n0 = int(y.sum()), int((1 - y).sum())
    if n1 == 0 or n0 == 0:
        return 0.5
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)
    ss = s[order]                                  # average ranks over ties
    i = 0
    while i < len(ss):
        j = i
        while j + 1 < len(ss) and ss[j + 1] == ss[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def best_threshold(logits, y):
    logits = np.asarray(logits, float)
    cands = np.unique(np.concatenate([[0.0], np.quantile(logits, np.linspace(0.02, 0.98, 97))]))
    scores = [macro_f1(y, logits > t) for t in cands]
    k = int(np.argmax(scores))
    return float(cands[k]), scores[k]


def calibrate(logits, y):
    """Threshold for macro-F1. Default 0 (the class-weighted loss already targets balanced
    classes). A dev-fitted threshold is used only if fitting on one half of dev and scoring
    the other half (both ways) beats the default."""
    logits, y = np.asarray(logits, float), np.asarray(y, int)
    f1_default = macro_f1(y, logits > 0)
    idx = np.arange(len(y))
    a = np.zeros(len(y), bool)
    for lab in (0, 1):                             # stratified halves
        ids = idx[y == lab]
        a[ids[::2]] = True
    pred = (logits > 0).astype(int)
    if a.any() and (~a).any():
        ta, _ = best_threshold(logits[a], y[a])
        tb, _ = best_threshold(logits[~a], y[~a])
        pred[~a] = logits[~a] > ta
        pred[a] = logits[a] > tb
    f1_cv = macro_f1(y, pred)
    t_all, f1_fit = best_threshold(logits, y)
    use = f1_cv > f1_default
    return (t_all if use else 0.0), {"dev_f1_threshold0": f1_default, "dev_f1_threshold_2fold": f1_cv,
                                     "dev_f1_threshold_fitted_on_dev": f1_fit, "threshold_used": round(t_all if use else 0.0, 4),
                                     "threshold_fitted": use}


# ---------------------------------------------------------------------------
# Adaptation methods
# ---------------------------------------------------------------------------
LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRMALinear(nn.Module):
    """LoRMA, additive-identity form: y = ((s B A) + I) W0 x = h + s B A h with h = W0 x.
    B = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class SVDLinear(nn.Module):
    """AdaLoRA: y = W0 x + s P diag(E) Q x. E (the 'singular values') starts at 0, so the
    layer starts as the Phase-1 layer; pruning a triplet sets its E entry to 0."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * self.E, self.P) * self.scale


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def apply_method(model, method, r=16, alpha=32):
    """Freeze the backbone, then open / insert what the method trains. Returns the number of
    backbone-side trainable parameters (the gradual-unfreezing controller opens blocks later)."""
    for p in model.parameters():
        p.requires_grad_(False)
    if method == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    elif method == "bitfit":
        n = 0
        for name, p in model.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad_(True)
                n += 1
        assert n, "BitFit needs bias terms; this backbone has none (use C)."
    elif method in ("lorma", "adalora"):
        names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in LINEAR_TARGETS]
        assert names, "no target linear layers found"
        for n in names:
            lin = model.get_submodule(n)
            _replace(model, n, LoRMALinear(lin, r, alpha) if method == "lorma" else SVDLinear(lin, r, alpha))
    elif method != "unfreeze":
        raise ValueError(method)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class AdaLoRA:
    """Importance-based rank allocation (Zhang et al. 2023). Importance of each weight =
    smoothed |w * grad| (EMA, beta1) times its smoothed uncertainty (EMA of the deviation,
    beta2); a triplet's score = score(E_i) + mean score of P[:, i] + mean score of Q[i, :].
    The total rank budget falls from r_init to r_target per layer on a cubic schedule between
    progress t_init and 1 - t_final, then stays fixed. Orthogonality penalty on P and Q."""

    def __init__(self, model, r_init, r_target, t_init=0.15, t_final=0.25, every=20, beta1=0.85, beta2=0.85, orth=0.1):
        self.mods = [(n, m) for n, m in model.named_modules() if isinstance(m, SVDLinear)]
        self.b0, self.bT = r_init * len(self.mods), r_target * len(self.mods)
        self.t_init, self.t_final, self.every = t_init, t_final, every
        self.beta1, self.beta2, self.orth = beta1, beta2, orth
        self.ipt, self.unc, self.final_mask = {}, {}, None

    def orth_loss(self):
        reg = 0.0
        for _, m in self.mods:
            r = m.E.shape[0]
            eye = torch.eye(r, device=m.P.device)
            reg = reg + torch.norm(m.P.float().t() @ m.P.float() - eye, p="fro") + torch.norm(m.Q.float() @ m.Q.float().t() - eye, p="fro")
        return self.orth * reg / max(1, len(self.mods))

    @torch.no_grad()
    def update_importance(self):
        for n, m in self.mods:
            for key, p in (("P", m.P), ("E", m.E), ("Q", m.Q)):
                if p.grad is None:
                    continue
                s = (p * p.grad).abs()
                k = (n, key)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(s), torch.zeros_like(s)
                self.ipt[k].mul_(self.beta1).add_((1 - self.beta1) * s)
                self.unc[k].mul_(self.beta2).add_((1 - self.beta2) * (s - self.ipt[k]).abs())

    def budget(self, p):
        if p < self.t_init:
            return self.b0
        if p >= 1 - self.t_final:
            return self.bT
        x = (p - self.t_init) / (1 - self.t_init - self.t_final)
        return int(self.bT + (self.b0 - self.bT) * (1 - x) ** 3)

    def _scores(self):
        out = []
        for n, m in self.mods:
            if (n, "E") not in self.ipt:
                out.append(torch.zeros_like(m.E))
                continue
            sc = lambda k: self.ipt[(n, k)] * self.unc[(n, k)]
            out.append(sc("E") + sc("P").mean(0) + sc("Q").mean(1))
        return out

    @torch.no_grad()
    def mask(self, progress, step):
        if progress < self.t_init:
            return
        if progress >= 1 - self.t_final:
            if self.final_mask is None:
                self.final_mask = self._prune(self.bT)
            for (_, m), keep in zip(self.mods, self.final_mask):
                m.E.mul_(keep)
            return
        if step % self.every == 0:
            self._prune(self.budget(progress))

    def _prune(self, budget):
        scores = self._scores()
        allv = torch.cat([s.flatten().float() for s in scores])
        if budget >= allv.numel():
            return [torch.ones_like(m.E) for _, m in self.mods]
        thr = torch.kthvalue(allv, allv.numel() - budget).values
        keeps = []
        for (_, m), s in zip(self.mods, scores):
            keep = (s.float() > thr).to(m.E.dtype)
            m.E.mul_(keep)
            keeps.append(keep)
        return keeps

    def ranks(self):
        return {n: int((m.E != 0).sum()) for n, m in self.mods}


class Unfreezer:
    """Gradual unfreezing from the top: at progress t, the top k blocks train
    (stages = ((t0, k0), (t1, k1), ...)); the final norm always trains."""

    def __init__(self, blocks, always, stages, log=print):
        self.blocks, self.stages, self.log, self.n_open, self.events = list(blocks), stages, log, 0, []
        for p in always:
            p.requires_grad_(True)

    def update(self, progress):
        n = max(k for t, k in self.stages if progress >= t)
        if n != self.n_open:
            for b in self.blocks[-n:]:
                for p in b.parameters():
                    p.requires_grad_(True)
            self.n_open = n
            self.events.append({"progress": round(progress, 3), "top_blocks_training": n})
            self.log(f"  gradual unfreezing: top {n} of {len(self.blocks)} blocks now train (progress {progress:.2f})")


# ---------------------------------------------------------------------------
# Encoders and models
# ---------------------------------------------------------------------------
def blocks_of(kind, model):
    return model.enc_blocks if kind == "D" else model.blocks


def final_norm(kind, model):
    return model.enc_norm if kind == "D" else model.norm_f


class AttnPool(nn.Module):
    """Attention pooling over tokens; the query starts at 0, i.e. as mean pooling."""

    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(d))

    def forward(self, h, mask):
        s = (h @ self.q) / math.sqrt(h.shape[-1])
        w = torch.softmax(s.masked_fill(~mask, -1e4), -1)
        return (w.unsqueeze(-1) * h).sum(1)


class ChunkEncoder(nn.Module):
    """token chunk [B, T] (right-padded) -> chunk vector [B, d]."""

    def __init__(self, kind, model, d):
        super().__init__()
        self.kind, self.m, self.d = kind, model, d
        self.pool = AttnPool(d)

    def states(self, ids, mask):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=ids.is_cuda):
            if self.kind == "D":
                h = self.m.encode(ids, mask)
            elif self.kind == "C":
                h = self.m.encode(ids, mode="prefix", prefix_len=mask.sum(1))
            else:                                   # A, B: left to right
                h = self.m.encode(ids)
        return h.float()

    def forward(self, ids, mask):
        return self.pool(self.states(ids, mask), mask)


class ChunkModel(nn.Module):
    def __init__(self, encoder, d, drop=0.1):
        super().__init__()
        self.enc, self.drop, self.out = encoder, nn.Dropout(drop), nn.Linear(d, 1)

    def forward(self, ids, mask):
        v = self.enc(ids, mask)
        return self.out(self.drop(v)).squeeze(-1), v


class Aggregator(nn.Module):
    """Case-level reader over chunk vectors X [B, n, d], positions P [B, n, 4]
    (start, end, is_first, is_last), mask M [B, n]. Returns (logit [B], chunk weights or None)."""

    KINDS = ("tail", "head", "mean", "max", "attn", "gru_attn", "transformer")

    def __init__(self, d, kind, h=256, drop=0.2):
        super().__init__()
        self.kind = kind
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, h), nn.GELU())
        self.pos = nn.Linear(4, h)
        if kind == "gru_attn":
            self.gru = nn.GRU(h, h // 2, batch_first=True, bidirectional=True)
        if kind == "transformer":
            layer = nn.TransformerEncoderLayer(h, 4, 2 * h, drop, batch_first=True, norm_first=True)
            self.tr = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
            self.cls = nn.Parameter(torch.zeros(1, 1, h))
        self.score = nn.Linear(h, 1)
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(h, 1))

    def forward(self, X, P, M):
        x = self.inp(X) + self.pos(P)
        B, n, _ = x.shape
        lens = M.sum(1)
        w = None
        if self.kind == "tail":
            z = x[torch.arange(B, device=x.device), lens - 1]
        elif self.kind == "head":
            z = x[:, 0]
        elif self.kind == "mean":
            m = M.unsqueeze(-1).float()
            z = (x * m).sum(1) / m.sum(1).clamp(min=1)
        elif self.kind == "max":
            z = x.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
        elif self.kind == "transformer":
            t = torch.cat([self.cls.expand(B, -1, -1), x], 1)
            pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=x.device), ~M], 1)
            z = self.tr(t, src_key_padding_mask=pad)[:, 0]
        else:
            if self.kind == "gru_attn":
                packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
                x = nn.utils.rnn.pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=n)[0]
            s = self.score(x).squeeze(-1).masked_fill(~M, -1e4)
            w = torch.softmax(s, -1)
            z = (w.unsqueeze(-1) * x).sum(1)
        return self.out(z).squeeze(-1), w


def pad_tokens(seqs, pad_id, device):
    L = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(np.asarray(s, dtype=np.int64))
        mask[i, :len(s)] = True
    return ids.to(device), mask.to(device)


def pos_feats(case, j):
    s, e, fs, fe = case["chunks"][j]
    return [fs, fe, float(j == 0), float(j == len(case["chunks"]) - 1)]


def region_select(case, region):
    """Chunk indices whose centre lies in the region ('begin' < 1/3, 'middle', 'end' > 2/3);
    if none does, the chunk whose centre is closest to the region."""
    lo, hi = {"begin": (0, 1 / 3), "middle": (1 / 3, 2 / 3), "end": (2 / 3, 1.0001)}[region]
    cent = [(fs + fe) / 2 for _, _, fs, fe in case["chunks"]]
    sel = [j for j, c in enumerate(cent) if lo <= c < hi]
    if not sel:
        sel = [int(np.argmin([min(abs(c - lo), abs(c - hi)) for c in cent]))]
    return sel


# ---------------------------------------------------------------------------
# Stage 1: adapt the backbone on single chunks
# ---------------------------------------------------------------------------
def lr_mult(p, warm=0.05):
    return min(1.0, p / warm) if p < warm else max(0.1, 1 - (p - warm) / (1 - warm))


def pick_chunk(case, rng):
    n = len(case["chunks"])
    return n - 1 if rng.random() < 0.5 else rng.randrange(n)


def build_net(exp, make_backbone, device, log):
    model, d, kind = make_backbone(exp["model"])
    n_backbone = sum(p.numel() for p in model.parameters())
    method = exp["method"]
    apply_method(model, method, exp.get("r", 16), exp.get("alpha", 32))
    net = ChunkModel(ChunkEncoder(kind, model, d), d).to(device)
    ada = unf = None
    head = list(net.enc.pool.parameters()) + list(net.out.parameters())
    groups = [{"params": head, "lr": exp["lr_head"], "base_lr": exp["lr_head"]}]
    if method == "unfreeze":
        blocks = list(blocks_of(kind, model))
        norm = list(final_norm(kind, model).parameters())
        unf = Unfreezer(blocks, norm, exp["stages"], log)
        groups.append({"params": norm, "lr": exp["lr"], "base_lr": exp["lr"]})
        k_max = max(k for _, k in exp["stages"])
        for i, b in enumerate(reversed(blocks[-k_max:])):        # i = 0 is the top block
            lr = exp["lr"] * exp.get("lr_decay", 0.85) ** i
            groups.append({"params": list(b.parameters()), "lr": lr, "base_lr": lr})
    else:
        bb = [p for p in model.parameters() if p.requires_grad]
        if bb:
            groups.append({"params": bb, "lr": exp["lr"], "base_lr": exp["lr"]})
    if method == "adalora":
        ada = AdaLoRA(net, exp["r"], exp["r_target"], orth=exp.get("orth", 0.1))
    return net, kind, n_backbone, groups, ada, unf


def trainable_count(net, unf, exp):
    if unf is not None:            # what eventually trains: head + final norm + top k_max blocks
        k_max = max(k for _, k in exp["stages"])
        blocks = unf.blocks[-k_max:]
        n = sum(p.numel() for b in blocks for p in b.parameters())
        return n + sum(p.numel() for p in net.parameters() if p.requires_grad and not any(p is q for b in blocks for q in b.parameters()))
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


@torch.no_grad()
def last_chunk_scores(net, cases, pad_id, device, bs):
    net.eval()
    out = []
    for i in range(0, len(cases), bs):
        part = cases[i:i + bs]
        seqs = [c["toks"][c["chunks"][-1][0]:c["chunks"][-1][1]] for c in part]
        ids, mask = pad_tokens(seqs, pad_id, device)
        out += net(ids, mask)[0].float().cpu().tolist()
    return np.array(out)


def train_stage1(exp, data, make_backbone, device, log):
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    net, kind, n_backbone, groups, ada, unf = build_net(exp, make_backbone, device, log)
    pad_id = data["pad_id"]
    by_id = {c["id"]: c for c in data["train"]}
    adapt = [by_id[i] for i in data["split"]["adapt"]]
    val = [by_id[i] for i in data["split"]["agg_val"]]
    y_val = [c["label"] for c in val]
    n_pos = sum(c["label"] for c in adapt)
    pos_weight = torch.tensor((len(adapt) - n_pos) / max(1, n_pos), device=device)
    opt = torch.optim.AdamW([g for g in groups], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_bs"], exp["accum"]
    planned = exp["epochs"] * max(1, math.ceil(len(adapt) / mb) // accum)
    budget = exp["budget_min"] * 60
    n_tr = trainable_count(net, unf, exp)
    n_head = sum(p.numel() for p in net.enc.pool.parameters()) + sum(p.numel() for p in net.out.parameters())
    pct = 100 * (n_tr - n_head) / n_backbone
    log(f"[{exp['name']}] backbone {exp['model']} {n_backbone / 1e6:.1f}M | method {exp['method']} | trainable "
        f"{n_tr - n_head:,} in the backbone ({pct:.2f}%) + {n_head:,} pooling/head | adapt cases {len(adapt)} (pos weight "
        f"{float(pos_weight):.2f}) | {planned} optimizer steps planned, budget {exp['budget_min']} min")
    t0, step, micro = time.time(), 0, 0
    st = {"best_auc": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]
    progress, done = 0.0, False
    params_all = [p for g in groups for p in g["params"]]
    tracked = {id(p) for p in params_all}

    def evaluate():
        auc = roc_auc(y_val, last_chunk_scores(net, val, pad_id, device, exp["infer_bs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 4),
                           "val_auc_last_chunk": round(auc, 4), "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | train loss {h['train_loss']:.4f} | "
            f"val AUC (last chunk) {auc:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        # AdaLoRA: only checkpoints whose rank budget is final can be chosen
        eligible = ada is None or progress >= 1 - ada.t_final
        if eligible and auc > st["best_auc"]:
            st["best_auc"] = auc
            st["best_state"] = {n: p.detach().cpu().clone() for n, p in net.named_parameters() if id(p) in tracked}

    for epoch in range(exp["epochs"]):
        order = list(range(len(adapt)))
        rng.shuffle(order)
        for i in range(0, len(order), mb):
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            if unf is not None:
                unf.update(progress)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [adapt[k] for k in order[i:i + mb]]
            seqs = []
            for c in part:
                j = pick_chunk(c, rng)
                s, e = c["chunks"][j][:2]
                seqs.append(c["toks"][s:e])
            ids, mask = pad_tokens(seqs, pad_id, device)
            y = torch.tensor([c["label"] for c in part], dtype=torch.float32, device=device)
            logit = net(ids, mask)[0].float()
            loss = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight)
            total = loss + (ada.orth_loss() if ada is not None else 0.0)
            scaler.scale(total / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            if ada is not None and all(torch.isfinite(p.grad).all() for p in params_all if p.grad is not None):
                ada.update_importance()
            torch.nn.utils.clip_grad_norm_([p for p in params_all if p.grad is not None], 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            if ada is not None:
                ada.mask(progress, step)
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points or st["best_state"] is None:      # epochs ran out just before the last evaluation point
        progress = 1.0
        if ada is not None:
            ada.mask(progress, step)
        evaluate()
    net.load_state_dict(st["best_state"], strict=False)
    info = {"n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": round(pct, 3),
            "stage1_minutes": round((time.time() - t0) / 60, 1), "stage1_steps": step, "stage1_history": st["hist"],
            "best_val_auc_last_chunk": round(st["best_auc"], 4),
            "peak_mem_train_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}
    if ada is not None:
        info["adalora_ranks"] = ada.ranks()
    if unf is not None:
        info["unfreeze_events"] = unf.events
    return net, info


# ---------------------------------------------------------------------------
# Stage 2: encode chunks
# ---------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(net, cases, pad_id, device, bs, last_k=None):
    """{case id: (vectors [n, d] fp16, stage-1 logits [n], positions [n, 4], chunk index list)}."""
    net.eval()
    items = []
    for ci, c in enumerate(cases):
        js = list(range(len(c["chunks"])))
        if last_k:
            js = js[-last_k:]
        items += [(ci, j) for j in js]
    items.sort(key=lambda t: -(cases[t[0]]["chunks"][t[1]][1] - cases[t[0]]["chunks"][t[1]][0]))
    vec, lg = {}, {}
    for i in range(0, len(items), bs):
        part = items[i:i + bs]
        seqs = [cases[ci]["toks"][cases[ci]["chunks"][j][0]:cases[ci]["chunks"][j][1]] for ci, j in part]
        ids, mask = pad_tokens(seqs, pad_id, device)
        logit, v = net(ids, mask)
        v, logit = v.half().cpu(), logit.float().cpu()
        for k, (ci, j) in enumerate(part):
            vec[(ci, j)], lg[(ci, j)] = v[k], float(logit[k])
    out = {}
    for ci, c in enumerate(cases):
        js = list(range(len(c["chunks"])))
        if last_k:
            js = js[-last_k:]
        out[c["id"]] = (torch.stack([vec[(ci, j)] for j in js]), np.array([lg[(ci, j)] for j in js]),
                        np.array([pos_feats(c, j) for j in js], dtype=np.float32), js)
    return out


# ---------------------------------------------------------------------------
# Stage 3: aggregators
# ---------------------------------------------------------------------------
def stack(enc, ids, cases_by_id, region=None, last_k=None):
    rows = []
    for i in ids:
        V, _, P, js = enc[i]
        sel = list(range(len(js)))
        if region:
            allowed = set(region_select(cases_by_id[i], region))
            sel = [k for k, j in enumerate(js) if j in allowed] or [len(js) - 1]
        if last_k:
            sel = sel[-last_k:]
        rows.append((V[sel].float(), torch.from_numpy(P[sel])))
    n = max(r[0].shape[0] for r in rows)
    d = rows[0][0].shape[1]
    X = torch.zeros(len(rows), n, d)
    Pm = torch.zeros(len(rows), n, 4)
    M = torch.zeros(len(rows), n, dtype=torch.bool)
    for b, (v, p) in enumerate(rows):
        X[b, :len(v)], Pm[b, :len(v)], M[b, :len(v)] = v, p, True
    return X, Pm, M


@torch.no_grad()
def agg_predict(agg, X, P, M, device, bs=256):
    agg.eval()
    logits, weights = [], []
    for i in range(0, len(X), bs):
        lg, w = agg(X[i:i + bs].to(device), P[i:i + bs].to(device), M[i:i + bs].to(device))
        logits.append(lg.float().cpu())
        weights.append(w.float().cpu() if w is not None else None)
    W = None if weights[0] is None else [w for part in weights for w in part]
    return torch.cat(logits).numpy(), W


def train_aggregator(kind, Xt, Pt, Mt, yt, Xv, Pv, Mv, yv, device, seed, epochs=40, patience=6, lr=1e-3, bs=64):
    torch.manual_seed(seed)
    agg = Aggregator(Xt.shape[-1], kind).to(device)
    opt = torch.optim.AdamW(agg.parameters(), lr=lr, weight_decay=0.01)
    y = torch.tensor(yt, dtype=torch.float32)
    pw = torch.tensor((len(yt) - sum(yt)) / max(1, sum(yt)), device=device)
    best, best_state, bad = -1.0, None, 0
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        agg.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            lg, _ = agg(Xt[b].to(device), Pt[b].to(device), Mt[b].to(device))
            loss = F.binary_cross_entropy_with_logits(lg, y[b].to(device), pos_weight=pw)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        auc = roc_auc(yv, agg_predict(agg, Xv, Pv, Mv, device)[0])
        if auc > best + 1e-5:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in agg.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    agg.load_state_dict(best_state)
    return agg, best


def attention_by_position(weights, enc, ids, bins=10):
    """Mean share of attention per decile of case position (chunk centre)."""
    acc = np.zeros(bins)
    for w, i in zip(weights, ids):
        P = enc[i][2]
        for k in range(len(P)):
            c = min(bins - 1, int((P[k, 0] + P[k, 1]) / 2 * bins))
            acc[c] += float(w[k])
    return (acc / max(1, len(ids))).round(4).tolist()


def stage3(exp, data, enc, device, log, out_dir, kd_enc=None):
    """Aggregator study + final case predictions. enc covers agg_train, agg_val, dev, test."""
    seed = exp["seed"]
    cases = {c["id"]: c for split in ("train", "dev", "test") for c in data[split]}
    tr_ids, va_ids = data["split"]["agg_train"], data["split"]["agg_val"]
    dev_ids, test_ids = [c["id"] for c in data["dev"]], [c["id"] for c in data["test"]]
    yt, yv, yd = [cases[i]["label"] for i in tr_ids], [cases[i]["label"] for i in va_ids], [cases[i]["label"] for i in dev_ids]
    rows, models = [], {}

    def report(name, val_scores, dev_scores, note=""):
        r = {"reader": name, "val_auc": round(roc_auc(yv, val_scores), 4), "dev_auc": round(roc_auc(yd, dev_scores), 4),
             "dev_macro_f1_at_0": macro_f1(yd, np.asarray(dev_scores) > 0), "note": note}
        rows.append(r)
        log(f"[{exp['name']}]   {name:28s} val AUC {r['val_auc']:.4f} | dev AUC {r['dev_auc']:.4f} | dev macro-F1 {r['dev_macro_f1_at_0']:.4f}")
        return r

    log(f"[{exp['name']}] stage 3: case readers (trained on {len(tr_ids)} held-out train cases, chosen on {len(va_ids)})")
    for name, f in (("stage1 last chunk only", lambda i: enc[i][1][-1]), ("stage1 mean of chunk logits", lambda i: enc[i][1].mean())):
        report(name, [f(i) for i in va_ids], [f(i) for i in dev_ids], "no training")
    full = {s: stack(enc, ids, cases) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids), ("test", test_ids))}
    for kind in Aggregator.KINDS:
        agg, _ = train_aggregator(kind, *full["tr"], yt, *full["va"], yv, device, seed)
        models[kind] = agg
        report(kind, agg_predict(agg, *full["va"], device)[0], agg_predict(agg, *full["dev"], device)[0], "whole case")
    for region in ("begin", "middle", "end"):
        st = {s: stack(enc, ids, cases, region=region) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids))}
        agg, _ = train_aggregator("attn", *st["tr"], yt, *st["va"], yv, device, seed)
        report(f"attn on {region} third only", agg_predict(agg, *st["va"], device)[0], agg_predict(agg, *st["dev"], device)[0], "section study")
    st = {s: stack(enc, ids, cases, last_k=KD_LAST) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids))}
    agg, _ = train_aggregator("attn", *st["tr"], yt, *st["va"], yv, device, seed)
    report(f"attn on last {KD_LAST} chunks", agg_predict(agg, *st["va"], device)[0], agg_predict(agg, *st["dev"], device)[0], "section study")

    best = max([r for r in rows if r["note"] == "whole case"], key=lambda r: r["val_auc"])
    chosen = models[best["reader"]]
    dev_logits = agg_predict(chosen, *full["dev"], device)[0]
    test_logits = agg_predict(chosen, *full["test"], device)[0]
    thr, cal = calibrate(dev_logits, yd)
    dev_pred = (dev_logits > thr).astype(int)
    test_pred = (test_logits > thr).astype(int)
    rep = binary_report(yd, dev_pred)
    log(f"[{exp['name']}] chosen reader: {best['reader']} (best val AUC {best['val_auc']:.4f}) | dev macro-F1 "
        f"{cal['dev_f1_threshold0']:.4f} at threshold 0, {cal['dev_f1_threshold_2fold']:.4f} with a 2-fold dev threshold "
        f"-> using threshold {thr:.3f} | final dev macro-F1 {rep['macro_f1']:.4f}")
    _, wdev = agg_predict(models["attn"], *full["dev"], device)
    att_pos = attention_by_position(wdev, enc, dev_ids)
    with open(Path(out_dir) / "readers.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    with open(Path(out_dir) / "dev_cases.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "label", "pred", "logit", "n_tokens", "n_chunks", "outcome"])
        for i, y, p, lg in zip(dev_ids, yd, dev_pred, dev_logits):
            c = cases[i]
            w.writerow([i, y, int(p), round(float(lg), 4), c["n_tok"], len(c["chunks"]),
                        "TP" if y and p else "TN" if not y and not p else "FP" if p else "FN"])
    errors = error_summary(yd, dev_pred, [cases[i]["n_tok"] for i in dev_ids])
    res = {"readers": rows, "chosen_reader": best["reader"], "calibration": cal, "dev_report": rep,
           "dev_auc": round(roc_auc(yd, dev_logits), 4), "attention_by_position_dev": att_pos, "errors_by_length": errors,
           "dev_logits": {i: float(v) for i, v in zip(dev_ids, dev_logits)},
           "test_logits": {i: float(v) for i, v in zip(test_ids, test_logits)}, "threshold": thr}
    if kd_enc is not None:                         # distillation targets on every train case's last chunks
        attn = models["attn"]
        targets = {}
        tr_all = [c["id"] for c in data["train"]]
        src = {i: (kd_enc[i] if i in kd_enc else enc[i]) for i in tr_all}
        for k in range(0, len(tr_all), 2048):
            ids = tr_all[k:k + 2048]
            X, P, M = stack(src, ids, cases, last_k=KD_LAST)
            lg, _ = agg_predict(chosen, X, P, M, device)
            _, wa = agg_predict(attn, X, P, M, device)
            for i, l, wv, m in zip(ids, lg, wa, M):
                targets[i] = {"logit": float(l), "attn": wv[:int(m.sum())].tolist()}
        torch.save(targets, Path(out_dir) / "kd_targets.pt")
        log(f"[{exp['name']}] wrote distillation targets for {len(targets)} train cases")
    return res, test_pred


def error_summary(y, pred, lengths):
    y, p, L = np.asarray(y), np.asarray(pred), np.asarray(lengths)
    qs = np.quantile(L, [0.25, 0.5, 0.75])
    out = []
    for k, (lo, hi) in enumerate(((0, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], np.inf))):
        m = (L >= lo) & (L < hi) if k < 3 else (L >= lo)
        if m.sum() == 0:
            continue
        out.append({"length_quartile": k + 1, "tokens_from": int(lo), "cases": int(m.sum()),
                    "macro_f1": macro_f1(y[m], p[m]), "false_pos": int(((p == 1) & (y == 0) & m).sum()),
                    "false_neg": int(((p == 0) & (y == 1) & m).sum())})
    return out


# ---------------------------------------------------------------------------
# Distillation student: reads the last KD_LAST chunks end to end
# ---------------------------------------------------------------------------
class HierStudent(nn.Module):
    def __init__(self, encoder, d):
        super().__init__()
        self.enc = encoder
        self.agg = Aggregator(d, "attn")

    def forward(self, cases, pad_id, device):
        seqs, owners, P = [], [], []
        for b, c in enumerate(cases):
            js = list(range(len(c["chunks"])))[-KD_LAST:]
            for j in js:
                s, e = c["chunks"][j][:2]
                seqs.append(c["toks"][s:e])
                owners.append(b)
                P.append(pos_feats(c, j))
        ids, mask = pad_tokens(seqs, pad_id, device)
        v = self.enc(ids, mask)
        n = max(owners.count(b) for b in range(len(cases)))
        X = torch.zeros(len(cases), n, v.shape[-1], device=device)
        Pm = torch.zeros(len(cases), n, 4, device=device)
        M = torch.zeros(len(cases), n, dtype=torch.bool, device=device)
        fill = [0] * len(cases)
        Pt = torch.tensor(P, device=device)
        for k, b in enumerate(owners):
            X[b, fill[b]], Pm[b, fill[b]], M[b, fill[b]] = v[k], Pt[k], True
            fill[b] += 1
        return self.agg(X, Pm, M)


def run_student(exp, data, make_backbone, device, log, out_dir, teacher_files):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    model, d, kind = make_backbone(exp["model"])
    apply_method(model, "full")
    n_backbone = sum(p.numel() for p in model.parameters())
    net = HierStudent(ChunkEncoder(kind, model, d), d).to(device)
    pad_id = data["pad_id"]
    teachers = [torch.load(f, weights_only=False) for f in teacher_files]
    T, alpha, beta = exp.get("kd_T", 2.0), exp.get("kd_alpha", 0.0), exp.get("attn_beta", 0.0)
    by_id = {c["id"]: c for c in data["train"]}
    val_ids = set(data["split"]["agg_val"])
    train = [c for c in data["train"] if c["id"] not in val_ids]
    val = [by_id[i] for i in data["split"]["agg_val"]]
    n_pos = sum(c["label"] for c in train)
    pos_weight = torch.tensor((len(train) - n_pos) / max(1, n_pos), device=device)
    enc_params = list(net.enc.m.parameters())
    groups = [{"params": enc_params, "lr": exp["lr"], "base_lr": exp["lr"]},
              {"params": list(net.enc.pool.parameters()) + list(net.agg.parameters()), "lr": exp["lr_head"], "base_lr": exp["lr_head"]}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_bs"], exp["accum"]
    planned = exp["epochs"] * max(1, math.ceil(len(train) / mb) // accum)
    budget = exp["budget_min"] * 60
    n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[{exp['name']}] student {exp['model']} ({n_backbone / 1e6:.1f}M, first {exp['student_layers']} layers of C) reads the "
        f"last {KD_LAST} chunks | teachers: {exp.get('teacher_names') or 'none (gold labels only)'} | alpha {alpha} T {T} "
        f"attention beta {beta} | {len(train)} cases, {planned} steps planned, budget {exp['budget_min']} min")
    t0, step, micro, progress, done = time.time(), 0, 0, 0.0, False
    st = {"best_auc": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]
    y_val = [c["label"] for c in val]

    def evaluate():
        auc = roc_auc(y_val, student_logits(net, val, pad_id, device, exp["infer_bs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 4),
                           "val_auc": round(auc, 4), "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | loss {h['train_loss']:.4f} | val AUC {auc:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        if auc > st["best_auc"]:
            st["best_auc"] = auc
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}

    for epoch in range(exp["epochs"]):
        order = list(range(len(train)))
        rng.shuffle(order)
        for i in range(0, len(order), mb):
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [train[k] for k in order[i:i + mb]]
            logit, w = net(part, pad_id, device)
            logit = logit.float()
            y = torch.tensor([c["label"] for c in part], dtype=torch.float32, device=device)
            loss = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight)
            if teachers and alpha > 0:
                p_t = torch.tensor([np.mean([1 / (1 + math.exp(-t[c["id"]]["logit"] / T)) for t in teachers]) for c in part],
                                   dtype=torch.float32, device=device)
                kd = F.binary_cross_entropy_with_logits(logit / T, p_t) * T * T
                loss = (1 - alpha) * loss + alpha * kd
            if teachers and beta > 0:
                kl = 0.0
                for b, c in enumerate(part):
                    a_t = torch.tensor(np.mean([t[c["id"]]["attn"] for t in teachers], 0), dtype=torch.float32, device=device)
                    a_s = w[b, :len(a_t)].float().clamp(min=1e-6)
                    kl = kl + (a_t * (a_t.clamp(min=1e-6).log() - a_s.log())).sum()
                loss = loss + beta * kl / len(part)
            scaler.scale(loss / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points:
        evaluate()
    best_auc, hist = st["best_auc"], st["hist"]
    net.load_state_dict(st["best_state"])
    train_min = round((time.time() - t0) / 60, 1)
    peak_train = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t1 = time.time()
    dev_logits = student_logits(net, data["dev"], pad_id, device, exp["infer_bs"])
    infer_s = time.time() - t1
    test_logits = student_logits(net, data["test"], pad_id, device, exp["infer_bs"])
    yd = [c["label"] for c in data["dev"]]
    thr, cal = calibrate(dev_logits, yd)
    rep = binary_report(yd, dev_logits > thr)
    info = {"exp": exp, "n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": 100.0, "train_minutes": train_min,
            "history": hist, "best_val_auc": round(best_auc, 4), "peak_mem_train_gb": peak_train,
            "peak_mem_infer_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0,
            "infer_cases_per_s": round(len(data["dev"]) / max(1e-6, infer_s), 2), "calibration": cal, "dev_report": rep,
            "dev_auc": round(roc_auc(yd, dev_logits), 4), "chosen_reader": f"student attn over last {KD_LAST} chunks",
            "threshold": thr, "dev_logits": {c["id"]: float(v) for c, v in zip(data["dev"], dev_logits)},
            "test_logits": {c["id"]: float(v) for c, v in zip(data["test"], test_logits)}}
    write_outputs(out_dir, info, data["test"], (test_logits > thr).astype(int), log)
    return info


@torch.no_grad()
def student_logits(net, cases, pad_id, device, bs):
    net.eval()
    out = []
    for i in range(0, len(cases), bs):
        out += net(cases[i:i + bs], pad_id, device)[0].float().cpu().tolist()
    return np.array(out)


# ---------------------------------------------------------------------------
# One standard experiment, outputs
# ---------------------------------------------------------------------------
def run_standard(exp, data, make_backbone, device, log, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    net, info = train_stage1(exp, data, make_backbone, device, log)
    by_id = {c["id"]: c for c in data["train"]}
    agg_cases = [by_id[i] for i in data["split"]["agg_train"] + data["split"]["agg_val"]]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    enc = encode_cases(net, data["dev"], data["pad_id"], device, exp["infer_bs"])
    dev_s = time.time() - t0
    info["infer_cases_per_s"] = round(len(data["dev"]) / max(1e-6, dev_s), 2)
    info["peak_mem_infer_gb"] = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    enc.update(encode_cases(net, data["test"], data["pad_id"], device, exp["infer_bs"]))
    enc.update(encode_cases(net, agg_cases, data["pad_id"], device, exp["infer_bs"]))
    kd_enc = None
    if exp.get("kd_teacher"):
        adapt = [by_id[i] for i in data["split"]["adapt"]]
        kd_enc = encode_cases(net, adapt, data["pad_id"], device, exp["infer_bs"], last_k=KD_LAST)
    info["encode_minutes"] = round((time.time() - t0) / 60, 1)
    log(f"[{exp['name']}] encoded chunks in {info['encode_minutes']} min ({info['infer_cases_per_s']} dev cases/s, all chunks)")
    del net
    if device.type == "cuda":
        torch.cuda.empty_cache()
    res, test_pred = stage3(exp, data, enc, device, log, out_dir, kd_enc)
    info.update(res)
    info["exp"] = exp
    write_outputs(out_dir, info, data["test"], test_pred, log)
    return info


def write_outputs(out_dir, info, test_cases, test_pred, log):
    out_dir = Path(out_dir)
    rows = []
    seen = set()
    for c, p in zip(test_cases, test_pred):
        assert c["id"] not in seen, f"duplicate test id {c['id']}"
        seen.add(c["id"])
        rows.append({"id": c["raw_id"], "pred": int(p)})
    assert len(rows) == len(test_cases) and all(r["pred"] in (0, 1) for r in rows)
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(rows, f)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    with open(out_dir / "probs.json", "w") as f:
        json.dump({"dev": info["dev_logits"], "test": info["test_logits"], "threshold": info["threshold"]}, f)
    slim = {k: v for k, v in info.items() if k not in ("dev_logits", "test_logits")}
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(slim, f, indent=1, default=str)
    r = info["dev_report"]
    log(f"[{info['exp']['name']}] DONE dev macro-F1 {r['macro_f1']:.4f} (P {r['macro_precision']:.4f} R {r['macro_recall']:.4f}) "
        f"| dev AUC {info['dev_auc']:.4f} | test predicted 1: {100 * float(np.mean(test_pred)):.1f}% -> {out_dir / 'submission.zip'}")


# ---------------------------------------------------------------------------
# Self-tests (tiny models, CPU, seconds)
# ---------------------------------------------------------------------------
def self_test(make_tiny, device, kinds=("D", "C", "A", "B")):
    torch.manual_seed(0)
    a, b = np.array([5, 9, 7, 11, 6, 8, 3, 2]), np.array([4, 10, 12])
    for kind in kinds:
        for method in ("lorma", "adalora", "unfreeze", "full") + (("bitfit",) if kind == "C" else ()):
            m, d = make_tiny(kind)
            enc = ChunkEncoder(kind, m, d).to(device).eval()
            ids, mask = pad_tokens([a, b], 0, device)
            with torch.no_grad():
                before = enc(ids, mask)
                one = enc(*pad_tokens([b], 0, device))
            assert (one[0] - before[1]).abs().max() < 1e-4, f"{kind}: padding changes the chunk vector"
            apply_method(m, method, r=4, alpha=8)
            enc = enc.to(device).eval()
            with torch.no_grad():
                after = enc(ids, mask)
            assert (before - after).abs().max() < 1e-4, f"{kind}/{method} does not start as the Phase-1 model"
            n_tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
            if method == "bitfit":
                assert n_tr > 0 and all(n.endswith(".bias") for n, p in m.named_parameters() if p.requires_grad)
            if method == "unfreeze":
                assert n_tr == 0
        # gradual unfreezing opens blocks from the top
        m, d = make_tiny(kind)
        apply_method(m, "unfreeze")
        blocks = list(blocks_of(kind, m))
        u = Unfreezer(blocks, final_norm(kind, m).parameters(), ((0, 1), (0.5, 2)), log=lambda *x: None)
        u.update(0.1)
        assert all(p.requires_grad for p in blocks[-1].parameters()) and not any(p.requires_grad for p in blocks[0].parameters())
        u.update(0.6)
        assert all(p.requires_grad for p in blocks[-2].parameters())
    # AdaLoRA: budget is reached and the forward still works
    k0 = kinds[-1]
    m, d = make_tiny(k0)
    apply_method(m, "adalora", r=4, alpha=8)
    net = ChunkModel(ChunkEncoder(k0, m, d), d).to(device)
    ada = AdaLoRA(net, 4, 2, t_init=0.1, t_final=0.3, every=1)
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=1e-2)
    for step in range(1, 21):
        ids, mask = pad_tokens([a, b], 0, device)
        loss = net(ids, mask)[0].pow(2).mean() + ada.orth_loss()
        opt.zero_grad()
        loss.backward()
        ada.update_importance()
        opt.step()
        ada.mask(step / 20, step)
    assert sum(ada.ranks().values()) <= ada.bT, ada.ranks()
    # aggregators: every kind runs; padding chunks do not change a case's logit
    X, P = torch.randn(2, 5, 16), torch.rand(2, 5, 4)
    M = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1]], dtype=torch.bool)
    for kind in Aggregator.KINDS:
        agg = Aggregator(16, kind, h=8).eval()
        lg, _ = agg(X, P, M)
        lg1, _ = agg(X[:1, :3], P[:1, :3], M[:1, :3])
        assert lg.shape == (2,) and abs(float((lg[0] - lg1[0]).detach())) < 1e-4, f"aggregator {kind} depends on padding"
    # student forward/backward
    ks = "C" if "C" in kinds else kinds[0]
    m, d = make_tiny(ks)
    apply_method(m, "full")
    st = HierStudent(ChunkEncoder(ks, m, d), d).to(device)
    fake = [{"toks": np.arange(1, 40, dtype=np.uint16) % 30 + 1, "chunks": [(0, 20, 0, .5), (10, 30, .25, .75), (20, 39, .5, 1)]},
            {"toks": np.arange(1, 9, dtype=np.uint16), "chunks": [(0, 8, 0, 1)]}]
    lg, w = st(fake, 0, device)
    lg.sum().backward()
    assert lg.shape == (2,) and w.shape == (2, 3)
    # calibration and metrics
    y = np.array([0, 0, 1, 1, 1, 0, 1, 0])
    s = np.array([-2, -1, 2, 1, 0.5, 0.2, -0.1, -3])
    assert abs(roc_auc(y, s) - 0.9375) < 1e-9
    thr, cal = calibrate(s, y)
    assert cal["dev_f1_threshold0"] == macro_f1(y, s > 0)
    print("cj_lib self-test passed: methods start as the Phase-1 model (LoRMA, AdaLoRA, BitFit, unfreezing, full) for "
          f"{', '.join(kinds)}; padding-safe chunk vectors; AdaLoRA reaches its rank budget; gradual unfreezing opens from the "
          "top; all 7 case readers are padding-safe; student trains; AUC / calibration ok.")


In [ ]:
%%writefile /kaggle/working/cj_train.py
"""cj_train.py -- one Phase-4 experiment: python cj_train.py <config.json> <experiment> <gpu index or -1>"""
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import cj_lib as CJ  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
data = torch.load(cfg["data"], weights_only=False)


def make_backbone(kind):
    """Phase-1 model from its checkpoint -> (module, d_model, kind used by the encoder)."""
    if kind == "D":
        import legal_d
        ck = torch.load(cfg["ckpt"]["D"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        m.dec_blocks = torch.nn.ModuleList()          # encoder only: the decoder is not used for classification
        m.grad_ckpt = exp.get("grad_ckpt", True)
        return m, ck["config"]["d_model"], "D"
    if kind in ("C", "Cs"):
        import c_model as mod
    elif kind == "A":
        import a_model as mod
    else:
        import b_model as mod
    ck = torch.load(cfg["ckpt"]["C" if kind == "Cs" else kind], map_location="cpu", weights_only=False)
    m = mod.build_model(ck["config"])
    m.load_state_dict(ck["model_state"])
    mod.USE_GRAD_CHECKPOINTING = exp.get("grad_ckpt", True)
    if kind == "Cs":                                  # distillation student: the first layers of C
        m.blocks = m.blocks[:exp["student_layers"]]
    return m, ck["config"]["d_model"], "C" if kind == "Cs" else kind


def log(*a):
    print(*a, flush=True)


out_dir = Path(cfg["out_dir"]) / name
if exp.get("type") == "student":
    files = [Path(cfg["out_dir"]) / t / "kd_targets.pt" for t in exp.get("teachers", [])]
    for f in files:
        assert f.exists(), f"missing teacher targets {f}"
    CJ.run_student(exp, data, make_backbone, device, log, out_dir, files)
else:
    CJ.run_standard(exp, data, make_backbone, device, log, out_dir)


In [ ]:
%%writefile /kaggle/working/cj_tfidf.py
"""cj_tfidf.py -- reference without any pretrained model (dev only, never submitted):
hashed word 1-2-gram TF-IDF + logistic regression on the same train cases.
python cj_tfidf.py <train.json> <dev.json> <out.json>"""
import json
import sys
import time

import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, sys.argv[4] if len(sys.argv) > 4 else ".")
import cj_lib as CJ  # noqa: E402

t0 = time.time()
train = [c for c in CJ.parse_cases(CJ.load_json_any(sys.argv[1])) if c["label"] is not None]
dev = CJ.parse_cases(CJ.load_json_any(sys.argv[2]))
res = {"cases_train": len(train), "cases_dev": len(dev)}
for name, cut in (("whole case", None), ("last 2,000 words", 2000)):
    pick = (lambda t: t) if cut is None else (lambda t: " ".join(t.split()[-cut:]))
    vec = HashingVectorizer(n_features=2 ** 20, ngram_range=(1, 2), alternate_sign=False, norm=None, lowercase=True)
    tf = TfidfTransformer(sublinear_tf=True)
    Xtr = tf.fit_transform(vec.transform(pick(c["text"]) for c in train))
    Xdv = tf.transform(vec.transform(pick(c["text"]) for c in dev))
    clf = LogisticRegression(C=4.0, class_weight="balanced", max_iter=3000, solver="liblinear")
    clf.fit(Xtr, [c["label"] for c in train])
    s = clf.decision_function(Xdv)
    y = [c["label"] for c in dev]
    rep = CJ.binary_report(y, s > 0)
    res[name] = {"dev_macro_f1": rep["macro_f1"], "dev_auc": round(CJ.roc_auc(y, s), 4), "report": rep}
    print(f"[TF-IDF] {name}: dev macro-F1 {rep['macro_f1']:.4f} | dev AUC {res[name]['dev_auc']:.4f} "
          f"({time.time() - t0:.0f}s)", flush=True)
json.dump(res, open(sys.argv[3], "w"), indent=1)


## 3. Data: download, clean, split, tokenize (T1)

In [ ]:
sys.path.insert(0, str(WORK_DIR))
import cj_lib as CJ  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"
from tokenizers import Tokenizer  # noqa: E402

t1 = Tokenizer.from_file(str(TOK_PATH))
PAD_ID = t1.token_to_id("<pad>") if t1.token_to_id("<pad>") is not None else 0
FALLBACK_ID = t1.token_to_id("<eos>") if t1.token_to_id("<eos>") is not None else PAD_ID


def encode_batch(texts):
    return [e.ids for e in t1.encode_batch(texts)]


def hf_file(*names):
    """First of the candidate paths that exists in the repo (the capstone doc names the first one)."""
    last = None
    for n in names:
        try:
            return hf_hub_download(REPO, n, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:      # noqa: BLE001
            last = e
    raise FileNotFoundError(f"none of {names} in {REPO}: {last}")


t0 = time.time()
RAW = {"train": hf_file("Phase_4_CJPE/Train/train.json", "Phase_4_CJPE/Train/train.jsonl"),
       "dev": hf_file("Phase_4_CJPE/Train/dev.json", "Phase_4_CJPE/Train/dev.jsonl"),
       "test": hf_file("Phase_4_CJPE/Test/test.json", "Phase_4_CJPE/Test/test.jsonl")}
cases = {s: CJ.parse_cases(CJ.load_json_any(p)) for s, p in RAW.items()}
print(f"downloaded + parsed in {time.time() - t0:.0f}s:", {s: len(v) for s, v in cases.items()})
for s in ("train", "dev"):
    assert all(c["label"] in (0, 1) for c in cases[s]), f"{s}: every case needs a 0/1 label"
test_ids = [c["id"] for c in cases["test"]]
assert len(test_ids) == len(set(test_ids)), "test: duplicate case ids"
# dev repeats some case ids: drop exact repeats (same id, text, label), rename the rest id#2, id#3, ...
seen_dev, n_used, dev_clean, n_rep = set(), {}, [], 0
for c in cases["dev"]:
    key = (c["id"], hashlib.md5(c["text"].encode("utf-8")).hexdigest(), c["label"])
    if key in seen_dev:
        n_rep += 1
        continue
    seen_dev.add(key)
    n = n_used.get(c["id"], 0)
    n_used[c["id"]] = n + 1
    dev_clean.append(dict(c, id=f"{c['id']}#{n + 1}") if n else c)
print(f"dev: {len(cases['dev'])} rows, {n_rep} exact repeats dropped, "
      f"{sum(1 for c in dev_clean if '#' in c['id'])} renamed -> {len(dev_clean)} dev cases")
cases["dev"] = dev_clean
print("example:", {k: (v[:120] + "...") if isinstance(v, str) else v for k, v in cases["train"][0].items()})

# exact duplicates: drop train cases whose text is also in dev/test, keep one copy of repeated train
# cases, drop texts that appear with both labels; make internal train ids unique
h = lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()
held = {h(c["text"]) for s in ("dev", "test") for c in cases[s]}
first, labels_of, n_overlap, n_dup = {}, {}, 0, 0
for c in cases["train"]:
    k = h(c["text"])
    if k in held:
        n_overlap += 1
        continue
    labels_of.setdefault(k, set()).add(c["label"])
    if k in first:
        n_dup += 1
        continue
    first[k] = c
conflict = {k for k, v in labels_of.items() if len(v) > 1}
train, seen_ids = [], {}
for k, c in first.items():
    if k in conflict:
        continue
    n = seen_ids.get(c["id"], 0)
    seen_ids[c["id"]] = n + 1
    if n:
        c = dict(c, id=f"{c['id']}#{n + 1}")
    train.append(c)
cases["train"] = train
print(f"train cleaning: {n_overlap} also in dev/test (dropped), {n_dup} duplicate copies (dropped), "
      f"{len(conflict)} texts with both labels (dropped) -> {len(train)} train cases")
for s in ("train", "dev"):
    y = [c["label"] for c in cases[s]]
    print(f"{s}: {len(y)} cases, {100 * np.mean(y):.1f}% label 1")
words = np.array([len(c["text"].split()) for c in cases["train"]])
print(f"train case length (words): median {np.median(words):.0f}, 90th pct {np.quantile(words, 0.9):.0f}, max {words.max()}")

t0 = time.time()
data = {"pad_id": PAD_ID, "tokenizer": "T1"}
for s in ("train", "dev", "test"):
    data[s] = CJ.tokenize_cases(cases[s], encode_batch, FALLBACK_ID)
data["split"] = CJ.split_train(data["train"])
ntok = np.array([c["n_tok"] for c in data["train"]])
nch = np.array([len(c["chunks"]) for c in data["train"]])
print(f"tokenized in {time.time() - t0:.0f}s | train tokens per case: median {np.median(ntok):.0f}, 90th pct "
      f"{np.quantile(ntok, 0.9):.0f} | chunks per case: mean {nch.mean():.1f}, max {nch.max()} | "
      f"{100 * np.mean([c['cut'] for c in data['train']]):.1f}% of cases longer than {CJ.MAX_TOKENS} tokens (middle skipped)")
print("split:", {k: len(v) for k, v in data["split"].items()})
DATA_PATH = OUT_DIR / "cj_data.pt"
torch.save(data, DATA_PATH)
DEV_LABELS = {c["id"]: c["label"] for c in data["dev"]}
json.dump(DEV_LABELS, open(OUT_DIR / "dev_labels.json", "w"))
del cases, train, first, data
gc.collect()


## 4. Self-tests (tiny models, a few seconds)

In [ ]:
import a_model  # noqa: E402
import b_model  # noqa: E402


def make_tiny(kind):
    if kind == "A":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=128)
        return a_model.build_model(cfg).eval(), 32
    if kind == "B":
        cfg = dict(vocab_size=64, d_model=32, n_layer=3, n_head=4, n_kv_head=1, d_head=8, d_ff=48, max_seq_len=128,
                   window=4, global_every=3, qk_norm=True, tokenizer_id="T1", attn_type="local+global", tied_embeddings=True)
        return b_model.build_model(cfg).eval(), 32
    raise KeyError(kind)


CJ.self_test(make_tiny, torch.device("cpu"), kinds=("A", "B"))


## 5. Experiment settings

In [ ]:
COMMON = dict(epochs=2, seed=13, lr_head=1e-3, grad_ckpt=True)
EXPERIMENTS = {
    "S3": dict(title="B + AdaLoRA", model="B", method="adalora", r=12, r_target=8, alpha=24, lr=1e-3, orth=0.1,
               micro_bs=8, accum=4, budget_min=70, infer_bs=32, submit=True),
    "S4": dict(title="A + gradual unfreezing", model="A", method="unfreeze", lr=5e-5, lr_decay=0.85,
               stages=[[0, 2], [0.2, 4], [0.4, 6], [0.6, 8]], micro_bs=8, accum=4, budget_min=80, infer_bs=24, submit=True),
    "R3": dict(title="B + LoRMA (reference for AdaLoRA)", model="B", method="lorma", r=16, alpha=32, lr=5e-4,
               micro_bs=8, accum=4, budget_min=70, infer_bs=32, submit=False),
}
RUN_ORDER = [("S3", []), ("S4", []), ("R3", [])]   # (run, must finish first)
CFG = {"work_dir": str(WORK_DIR), "data": str(DATA_PATH), "ckpt": CKPT, "out_dir": str(OUT_DIR),
       "common": COMMON, "experiments": EXPERIMENTS}
CONFIG_PATH = WORK_DIR / "cj_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
for name, e in EXPERIMENTS.items():
    print(f"{name}: {e['title']}")


## 6. Run (two at a time, one per GPU)

In [ ]:
# reference without pretraining, on CPU while the GPUs train
tfidf_log = open(OUT_DIR / "tfidf.log", "w")
tfidf = subprocess.Popen([sys.executable, str(WORK_DIR / "cj_tfidf.py"), RAW["train"], RAW["dev"],
                          str(OUT_DIR / "tfidf.json"), str(WORK_DIR)], stdout=tfidf_log, stderr=subprocess.STDOUT)


def stream(proc, tag):
    for line in proc.stdout:
        print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
t_all = time.time()
while queue or running:
    for slot in slots:
        if slot in running:
            continue
        ready = [q for q in queue if all(d in codes for d in q[1])]
        if not ready:
            continue
        name, deps = ready[0]
        queue.remove(ready[0])
        if any(codes.get(d) != 0 for d in deps):
            print(f">>> skipping {name}: {deps} did not all finish", flush=True)
            codes[name] = -1
            continue
        env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        proc = subprocess.Popen([sys.executable, str(WORK_DIR / "cj_train.py"), str(CONFIG_PATH), name, str(slot)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        th = threading.Thread(target=stream, args=(proc, name), daemon=True)
        th.start()
        running[slot] = (name, proc, th)
        print(f">>> started {name} ({EXPERIMENTS[name]['title']}) on {'GPU ' + str(slot) if slot >= 0 else 'CPU'}", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - t_all) / 60:.1f} min so far)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)
tfidf.wait()
print(open(OUT_DIR / "tfidf.log").read()[-2000:])


## 7. Results, analysis tables, submissions

In [ ]:
rows, by = [], {}
for name, _ in RUN_ORDER:
    mpath = OUT_DIR / name / "metrics.json"
    if not mpath.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mpath))
    e = EXPERIMENTS[name]
    r = {"run": name, "title": e["title"], "model": e["model"], "method": e.get("method", "distillation" if e.get("teachers") else "student"),
         "backbone_M": round(m["n_backbone"] / 1e6, 1), "backbone_params": m["n_backbone"], "trainable": m["n_trainable"], "trainable_pct": m["trainable_pct"],
         "dev_macro_f1": m["dev_report"]["macro_f1"], "dev_precision": m["dev_report"]["macro_precision"],
         "dev_recall": m["dev_report"]["macro_recall"], "dev_auc": m["dev_auc"], "reader": m["chosen_reader"],
         "threshold": m["threshold"], "f1_threshold0": m["calibration"]["dev_f1_threshold0"],
         "train_min": m.get("stage1_minutes", m.get("train_minutes")), "encode_min": m.get("encode_minutes", 0),
         "peak_mem_train_gb": m["peak_mem_train_gb"], "peak_mem_infer_gb": m["peak_mem_infer_gb"],
         "infer_cases_per_s": m["infer_cases_per_s"], "submit": e["submit"]}
    rows.append(r)
    by[name] = r
assert rows, "No run finished -- read the error lines above."
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print(f"{'run':4s} {'title':42s} {'trainable':>12s} {'%':>7s} {'F1':>7s} {'P':>7s} {'R':>7s} {'AUC':>7s} {'min':>6s} {'GB':>5s} {'case/s':>7s}")
for r in rows:
    print(f"{r['run']:4s} {r['title'][:42]:42s} {r['trainable']:>12,} {r['trainable_pct']:7.2f} {r['dev_macro_f1']:7.4f} "
          f"{r['dev_precision']:7.4f} {r['dev_recall']:7.4f} {r['dev_auc']:7.4f} {r['train_min'] + r['encode_min']:6.1f} "
          f"{r['peak_mem_train_gb']:5.1f} {r['infer_cases_per_s']:7.2f}")

# long-document readers and the section study, per run (dev numbers are for reporting; choice used val AUC)
for name in by:
    m = json.load(open(OUT_DIR / name / "metrics.json"))
    if "readers" not in m:
        continue
    print(f"\n{name} -- case readers (chosen: {m['chosen_reader']}):")
    for x in m["readers"]:
        print(f"  {x['reader']:30s} val AUC {x['val_auc']:.4f} | dev AUC {x['dev_auc']:.4f} | dev macro-F1 {x['dev_macro_f1_at_0']:.4f}  ({x['note']})")
    print("  attention share by case position (deciles, start -> end):", m["attention_by_position_dev"])
    print("  errors by case length:", m["errors_by_length"])

# AdaLoRA: where the rank budget went (sum of kept ranks per block, attention vs MLP)
for n in ("S3",):
    if (OUT_DIR / n / "metrics.json").exists():
        ranks = json.load(open(OUT_DIR / n / "metrics.json")).get("adalora_ranks", {})
        per = {}
        for name, r in ranks.items():
            blk = int(name.split("blocks.")[1].split(".")[0])
            part = "attn" if any(t in name for t in ("q_proj", "k_proj", "v_proj", "o_proj")) else "mlp"
            per.setdefault(blk, {"attn": 0, "mlp": 0})[part] += r
        print("\nAdaLoRA kept rank per block (attention / MLP):")
        print("  " + " | ".join(f"{b}: {v['attn']}/{v['mlp']}" for b, v in sorted(per.items())))
for n in ("S4",):
    if (OUT_DIR / n / "metrics.json").exists():
        print("\nGradual unfreezing:", json.load(open(OUT_DIR / n / "metrics.json")).get("unfreeze_events"))
if (OUT_DIR / "tfidf.json").exists():
    tf = json.load(open(OUT_DIR / "tfidf.json"))
    print("\nTF-IDF + logistic regression (no pretraining):",
          {k: (v["dev_macro_f1"], v["dev_auc"]) for k, v in tf.items() if isinstance(v, dict)})

ens = {}
for name in by:
    p = json.load(open(OUT_DIR / name / "probs.json"))
    ens[name] = dict(p, title=EXPERIMENTS[name]["title"], dev_macro_f1=by[name]["dev_macro_f1"])
ens["dev_labels"] = json.load(open(OUT_DIR / "dev_labels.json"))
json.dump(ens, open(OUT_DIR / "ens_nb2.json", "w"))
print("\nSubmit these (Output tab):")
for name, e in EXPERIMENTS.items():
    if e["submit"]:
        z = OUT_DIR / name / "submission.zip"
        print(f"  {name} ({e['title']}): {z}" if z.exists() else f"  {name}: MISSING")
print(f"Also download {OUT_DIR / 'ens_nb2.json'} for the ensemble (S6).")
